# 🧪 PT-W1-D3 概念实验：Aggregate 边界判定

> 配套阅读：同名 .md
> 实验目标：用 dataclass 模拟 5 个 P0 Aggregate，验证不变量和事务边界

## 第 1 格：定义 Entity / Value Object / Aggregate 基础类型

In [ ]:
from dataclasses import dataclass, field
from typing import List

@dataclass(frozen=True)
class ValueObject:
    name: str
    properties: tuple

@dataclass
class Entity:
    identity: str
    entity_type: str
    state: str

@dataclass
class Aggregate:
    root: Entity
    value_objects: List[ValueObject] = field(default_factory=list)
    invariants: List[str] = field(default_factory=list)
    internal_entities: List[Entity] = field(default_factory=list)

# 演示 Entity vs Value Object
space_attrs = ValueObject("PhysicalAttributes", ("面积=50㎡", "楼层=L2"))
print("ValueObject:", space_attrs)
print("  两个 VO 属性相同就是'相等'，无 ID")

e1 = Entity("B1-F2-015", "ResourceUnit", "空置")
e2 = Entity("B1-F2-015", "ResourceUnit", "已租")
print("\nEntity:", e1)
print("  ID 不变，state 可变:", e1.identity == e2.identity, e1.state != e2.state)

## 第 2 格：5 个 P0 Aggregate 定义

In [ ]:
aggregates = {
    "Asset Foundation": Aggregate(
        root=Entity("ResourceUnit-*", "ResourceUnit", "Active"),
        value_objects=[
            ValueObject("ResourceType", ("商业", "办公", "仓储")),
            ValueObject("PhysicalAttributes", ("面积", "工程条件")),
        ],
        invariants=["编码全局唯一不可变", "层级关系不可循环", "Asset 不拥有商业可用性"],
    ),
    "Lease/Occupancy": Aggregate(
        root=Entity("Occupancy-*", "Occupancy", "Active"),
        value_objects=[
            ValueObject("OccupancyPeriod", ("交付日", "退场日")),
            ValueObject("AvailabilityAssessment", ("组合判断结果",)),
        ],
        invariants=["一个 ResourceUnit 同一时间只有一个 Active Occupancy", "出租率 ≠ 占用率"],
    ),
    "Contract Lifecycle": Aggregate(
        root=Entity("Contract-*", "Contract", "Active"),
        value_objects=[
            ValueObject("ClauseType", ("固定", "面积", "提成", "保底取高")),
            ValueObject("RentMethod", ("12种计租矩阵",)),
        ],
        invariants=["条款定义归此 Context", "合同状态五态封闭"],
        internal_entities=[Entity("Clause-*", "ContractClause", "生效")],
    ),
    "Billing & AR": Aggregate(
        root=Entity("Bill-*", "Bill", "已生效"),
        value_objects=[
            ValueObject("FeeItem", ("租金", "物业费", "推广费")),
            ValueObject("ArStatus", ("未生效", "已生效", "已核销")),
        ],
        invariants=["消费 Contract Clause 执行算费", "已终止合同零账单"],
    ),
    "Merchant": Aggregate(
        root=Entity("Merchant-*", "Merchant", "经营中"),
        value_objects=[
            ValueObject("LegalIdentity", ("统一社会信用代码",)),
            ValueObject("OperatingStatus", ("正常", "监管", "禁止")),
        ],
        invariants=["名称+LegalIdentity 唯一", "Merchant 是 Identity Reference 主体"],
    ),
}

for name, agg in aggregates.items():
    print(f"\n📦 {name}")
    print(f"   Root: {agg.root.entity_type} (id={agg.root.identity})")
    print(f"   VOs:  {[vo.name for vo in agg.value_objects]}")
    print(f"   不变量: {agg.invariants[0]}")

## 第 3 格：Aggregate 边界 = AI 推理边界

In [ ]:
effect_registry = {
    "occupancy": {"trigger_agg": "Contract Lifecycle", "target_agg": "Asset Foundation"},
    "financial": {"trigger_agg": "Contract Lifecycle", "target_agg": "Billing & AR"},
    "state-transition": {"trigger_agg": "*", "target_agg": "*"},
}

cross_agg_queries = {
    "铺位面积": ["Asset Foundation"],
    "这个铺位能出租吗": ["Lease/Occupancy", "Asset Foundation"],
    "合同终止后影响什么": ["Contract Lifecycle", "Lease/Occupancy", "Billing & AR"],
}

for q, needed in cross_agg_queries.items():
    print()
    print(f"查询: 【{q}】")
    if len(needed) == 1:
        print(f"  ✅ 可在 {needed[0]} 内推理")
    else:
        print(f"  ⚠️ 需跨 Aggregate: {needed}")
        print(f"  → 跨边界推理需要 effect-registry 桥接")

## 第 4 格：AvailabilityAssessment 是规则不是字段

In [ ]:
def available_for_lease(space, active_occupancy, restrictions):
    """组合规则判断铺位可出租性 — 这是 Rule 不是 Entity 属性"""
    checks = [
        ("Asset 状态", space.state == "Active"),
        ("无 Active Occupancy", active_occupancy is None),
        ("无 Restriction", not restrictions),
    ]
    result = all(v for _, v in checks)
    print("AvailableForLeasing 判断：")
    for name, passed in checks:
        print(f"  {'✅' if passed else '❌'} {name}: {passed}")
    print(f"  → 可出租: {result}")
    return result

class Space:
    state: str = "Active"

print("\n⚠️ AI Agent 不能'写入'可用性状态，只能通过查规则推断")
print("这是 Ontology 的 Rule 维度，不是 Entity 属性")

## 第 5 格：可视化 Aggregate 边界与跨聚合 effect

In [ ]:
from matplotlib import font_manager, pyplot as plt
import numpy as np

font_path = "/usr/share/fonts/opentype/noto/NotoSansCJK-Regular.ttc"
font_manager.fontManager.addfont(font_path)
font_name = font_manager.FontProperties(fname=font_path).get_name()
plt.rcParams["font.family"] = font_name
plt.rcParams["axes.unicode_minus"] = False
print("字体:", font_name)
import matplotlib.patches as mpatches

agg_names = ["Asset Foundation", "Lease/Occupancy", "Contract Lifecycle", "Billing & AR", "Merchant"]
positions = [(1,3),(3,3),(5,3),(2,1),(4,1)]
colors_agg = ["#e74c3c","#3498db","#2ecc71","#f39c12","#9b59b6"]

fig, ax = plt.subplots(figsize=(10, 6))
for i, (name, (x, y)) in enumerate(zip(agg_names, positions)):
    rect = mpatches.FancyBboxPatch((x-0.9, y-0.5), 1.8, 1.0,
            boxstyle="round,pad=0.1", facecolor=colors_agg[i], alpha=0.7)
    ax.add_patch(rect)
    ax.text(x, y, name, ha="center", va="center", fontsize=8, fontweight="bold")

effect_edges = [
    ((3,3),(1,3),"occupancy"), ((3,3),(2,1),"financial"), ((3,3),(3,3),"occupancy"),
]
for (sx,sy),(tx,ty),label in effect_edges:
    if (sx,sy) == (tx,ty):
        continue
    ax.annotate("", xy=(tx,ty), xytext=(sx,sy),
                arrowprops=dict(arrowstyle="->", color="#2c3e50", lw=2, ls="--"))
    ax.text((sx+tx)/2, (sy+ty)/2+0.2, label, fontsize=7, color="#e74c3c", style="italic")

ax.set_xlim(-0.5, 6.5); ax.set_ylim(0, 4.5)
ax.axis("off")
ax.set_title("Aggregate 边界与跨聚合 Effect", fontsize=14)
plt.tight_layout()
plt.savefig("/tmp/w1d3_aggregates.png", dpi=120)
plt.show()
print("结论：Aggregate 边界 = Agent 推理边界，effect-registry = 跨边界桥梁")